# Eight Puzzle Solver Using Search Algorithms

The Eight Puzzle is a sliding puzzle consisting of a 3×3 board with eight numbered tiles and one empty cell, represented by `0`.

The objective is to reach a predefined goal configuration by moving tiles into the empty cell. This project implements three search algorithms to solve the puzzle: **Breadth-First Search (BFS)**, **Depth-First Search (DFS)**, and **Bidirectional Search**.

## Puzzle Representation and State Operations

The puzzle is represented as a 3×3 matrix, where each configuration corresponds to a state. The `Eight_puzzle` class provides methods for generating initial configurations, checking solvability, identifying legal moves, applying actions, and testing whether the goal state has been reached.

The solvability of a configuration is determined by counting the number of inversions. The goal configuration is `[[0, 1, 2], [3, 4, 5], [6, 7, 8]]`.

## Node Representation

The `Node` class represents a state in the search tree. Each node stores the current puzzle configuration, a reference to its parent node, and its depth (`d`). Parent references are used to reconstruct the solution path after reaching the goal state.

## Search Algorithms

**Breadth-First Search (BFS)** explores the state space level by level using a queue. Since all moves have equal cost, BFS can find a shortest solution path.

**Depth-First Search (DFS)** explores one branch as deeply as possible before backtracking. A depth limit is used to restrict the search. Instead of maintaining a global explored set, the implementation checks parent nodes to avoid revisiting states along the same path.

**Bidirectional Search** explores the state space from both the initial and goal configurations. It checks for a common state between the two search directions and reconstructs the solution path when the searches meet.

## Implementation and Execution

The implementation combines the puzzle representation, node structure, and three search algorithms. The initial puzzle configuration can be generated randomly or assigned manually.

In the example below, the same initial configuration is provided to all three solvers. The number of explored nodes and the solution depth are reported for each algorithm. The Bidirectional Search implementation also prints the resulting solution path.

In [ ]:
from copy import deepcopy
from random import *
import copy


# ============================================================
# Node Representation
# ============================================================

class Node:
    def __init__(self, puzzle, parent=None, d=0):
        self.puzzle = puzzle
        self.parent = parent
        self.d = d

    def __eq__(self, rhs):
        if rhs == None:
            return False
        return self.puzzle == rhs.puzzle

    def print_path(self):
        if type(self.parent) != type(None):
            self.parent.print_path()
        self.puzzle.print_table()
        print('')

    def reverse_print_path(self):
        self.puzzle.print_table()
        if type(self.parent) != type(None):
            self.parent.print_path()
        print('')


# ============================================================
# Eight Puzzle Representation
# ============================================================

class Eight_puzzle:
    def __init__(self, n=100):
        self.table = [[0, 1, 2], [3, 4, 5], [6, 7, 8]]

        for i in range(n):
            actions = self.get_legal()
            action = choice(actions)
            self.apply_action(action)

    def __eq__(self, rhs):
        return self.table == rhs.table

    def is_solvable(self):
        inversions = 0

        for i in range(9):
            for j in range(i + 1, 9):
                if self.table[i // 3][i % 3] == 0 or self.table[j // 3][j % 3] == 0:
                    continue

                if self.table[i // 3][i % 3] > self.table[j // 3][j % 3]:
                    inversions += 1

        if inversions % 2 == 0:
            return True
        else:
            return False

    def apply_action(self, action):
        x, y = action[0], action[1]

        if self.table[x][y] != 0:
            x2, y2 = None, None

            for sum in [(-1, 0), (0, -1), (0, 1), (1, 0)]:
                if x + sum[0] >= 0 and x + sum[0] < 3 and y + sum[1] >= 0 and y + sum[1] < 3 and self.table[x + sum[0]][y + sum[1]] == 0:
                    x2, y2 = x + sum[0], y + sum[1]
                    break

            if x2 != None and y2 != None:
                self.table[x][y], self.table[x2][y2] = self.table[x2][y2], self.table[x][y]

    def is_goal(self):
        return self.table == [[0, 1, 2], [3, 4, 5], [6, 7, 8]]

    def copy_puzzle(self):
        puzzle2 = Eight_puzzle(0)
        puzzle2.table = self.copy_table()
        return puzzle2

    def copy_table(self):
        return copy.deepcopy(self.table)

    def get_legal(self):
        res = []
        flg = False

        for i in range(3):
            if flg:
                break

            for j in range(3):
                if self.table[i][j] == 0:
                    x, y = i, j
                    flg = True
                    break

        for sum in [(-1, 0), (0, -1), (0, 1), (1, 0)]:
            if x + sum[0] >= 0 and x + sum[0] < 3 and y + sum[1] >= 0 and y + sum[1] < 3:
                res.append((x + sum[0], y + sum[1]))

        return res

    def print_table(self):
        for i in range(3):
            for j in range(3):
                print(self.table[i][j], end=' ')
            print('')


# ============================================================
# Breadth-First Search (BFS)
# ============================================================

class BFS_solver:
    def __init__(self):
        self.frontier = []
        self.explored_set = []
        self.goal = None

    def get_explored(self):  # number of explored nodes
        return len(self.explored_set)

    def solve(self, puzzle):  # solves the puzzle and returns the goal node (with its parents our desired path is found) or returns None if no goal was found
        if not puzzle.is_solvable():
            return None

        self.frontier.append(Node(puzzle.copy_puzzle()))

        if self.frontier[0].puzzle.is_goal():
            self.goal = self.frontier[0]
            return self.frontier[0]

        while len(self.frontier) != 0:
            cur = self.frontier[0]  # first in last out (queue)
            del(self.frontier[0])
            self.explored_set.append(cur)

            for a in cur.puzzle.get_legal():
                suc = Node(cur.puzzle.copy_puzzle(), cur, cur.d + 1)
                suc.puzzle.apply_action(a)

                if suc.puzzle.is_goal():  # goal is found
                    self.goal = suc
                    return suc

                if not suc in self.frontier and not suc in self.explored_set:  # not yet encountered
                    self.frontier.append(suc)

        return None


# ============================================================
# Depth-First Search (DFS)
# ============================================================

def is_in_parents(a, b):
    while b != None:
        if a == b:
            return True
        b = b.parent
    return False


class DFS_solver:
    """
    no explored set was used intentionally, because by using it we would fail the advantage of dfs which is
    memory usage of O(md) instead is_in_parents() was used to avoid encountering a same node on a same path
    """

    def __init__(self):
        self.frontier = []
        self.explored = 0
        self.goal = None

    def get_explored(self):
        return self.explored

    def solve(self, puzzle, d=10):
        if not puzzle.is_solvable():
            return None

        self.frontier.append(Node(puzzle.copy_puzzle()))

        if self.frontier[0].puzzle.is_goal():
            self.goal = self.frontier[0]
            return self.frontier[0]

        while len(self.frontier) != 0:
            cur = self.frontier[-1]
            del(self.frontier[-1])
            self.explored += 1

            for a in cur.puzzle.get_legal():
                suc = Node(cur.puzzle.copy_puzzle(), cur, cur.d + 1)
                suc.puzzle.apply_action(a)

                if suc.puzzle.is_goal():
                    self.goal = suc
                    return suc

                if not suc in self.frontier and not is_in_parents(suc, cur) and suc.d <= d:
                    self.frontier.append(suc)

        return None


# ============================================================
# Bidirectional Search
# ============================================================

class Bidirectional_solver:
    def __init__(self):
        self.forw_frontier = []
        self.forw_explored_set = []
        self.back_frontier = []
        self.back_explored_set = []
        self.goal = None
        self.initial = None

    def get_explored(self):
        return len(self.forw_explored_set) + len(self.back_explored_set)

    def construct_goal(self, a, b):
        cur = b.parent
        pre = a

        while cur != None:
            pre = Node(cur.puzzle, pre, pre.d + 1)
            cur = cur.parent

        self.goal = pre

    def solve(self, puzzle):
        if not puzzle.is_solvable():
            return None

        self.forw_frontier.append(Node(puzzle.copy_puzzle()))
        self.initial = puzzle.copy_puzzle()
        self.back_frontier.append(Node(Eight_puzzle(0)))

        if self.forw_frontier[0].puzzle.is_goal():
            self.goal = self.forw_frontier[0]
            return self.forw_frontier[0]

        while len(self.forw_frontier) != 0:
            cur = self.forw_frontier[0]
            del(self.forw_frontier[0])
            self.forw_explored_set.append(cur)

            for a in cur.puzzle.get_legal():
                suc = Node(cur.puzzle.copy_puzzle(), cur, cur.d + 1)
                suc.puzzle.apply_action(a)

                if suc.puzzle.is_goal():
                    self.goal = suc
                    return suc

                if not suc in self.forw_frontier and not suc in self.forw_explored_set:
                    self.forw_frontier.append(suc)

            cur = self.back_frontier[0]
            del(self.back_frontier[0])
            self.back_explored_set.append(cur)

            for a in cur.puzzle.get_legal():
                suc = Node(cur.puzzle.copy_puzzle(), cur)
                suc.puzzle.apply_action(a)

                if suc.puzzle == self.initial:
                    self.construct_goal(self.initial, suc)
                    return self.goal

                if not suc in self.back_frontier and not suc in self.back_explored_set:
                    self.back_frontier.append(suc)

            # Check shared states:
            for i in self.forw_frontier:
                for j in self.back_frontier:
                    if i == j:
                        self.construct_goal(i, j)
                        return self.goal

            for i in self.forw_frontier:
                for j in self.back_explored_set:
                    if i == j:
                        self.construct_goal(i, j)
                        return self.goal

            for i in self.back_frontier:
                for j in self.forw_explored_set:
                    if i == j:
                        self.construct_goal(i, j)
                        return self.goal

        return None


# ============================================================
# Main Execution
# ============================================================

puzzle = Eight_puzzle(30)
puzzle.table = [[4, 3, 1], [7, 2, 8], [5, 6, 0]]  # table can be set manually

print('puzzle :')
puzzle.print_table()
print('')

solver1 = BFS_solver()
solver2 = DFS_solver()
solver3 = Bidirectional_solver()

ans1 = solver1.solve(puzzle)
ans2 = solver2.solve(puzzle)
ans3 = solver3.solve(puzzle)

if ans1 != None:
    print('1 explored:', solver1.get_explored())
    print('1 length:', ans1.d)

if ans2 != None:
    print('2 explored:', solver2.get_explored())
    print('2 length:', ans2.d)

if ans3 != None:
    print('3 explored', solver3.get_explored())
    print('3 length', ans3.d)
    print('3 result:')
    ans3.print_path()